# Pressmint OCR inference evaluations

This notebook compare various OCR inference methods, primarily to be compared against transkribus und ground truth.

# setup

## colab setup

In [ ]:
# colab specifics
enable_colab = False
if enable_colab:
    import os

    # connect to drive folder
    from google.colab import drive

    drive.mount("/content/drive")

    print(os.listdir("/data"))
    print(os.listdir("/content/drive/MyDrive/pressmint-ocr-comparisons/"))
    !ln -s /content/drive/MyDrive/pressmint-ocr-comparisons/data/ /data
    print(os.listdir("/data"))

    # install dependencies
    !pip install jiwer==4.0.0
    !pip install Levenshtein==0.27.1
    !pip install openai==1.97.1
    !pip install pandas==2.2.3
    !pip install google-cloud-vision==3.10.2
    !pip install google-generativeai==0.8.5
    !pip install google-cloud-aiplatform==1.105.0
    !pip install anthropic==0.59.0
    !pip install requests==2.32.4
    !pip install plotly==6.1.1
    !pip install tabulate==0.9.0
    !pip install kaleido==0.2.1
    !pip install pillow==11.3.0

## dependencies

In [ ]:
import base64
import difflib
import getpass
import io
import json
import os
import pickle
import time
from pathlib import Path
from typing import Callable

import anthropic
import Levenshtein
import openai
import pandas as pd
import plotly.graph_objects as go
import vertexai
from google.cloud import vision
from google.oauth2 import service_account
from jiwer import cer, wer
from PIL import Image
from vertexai.preview.generative_models import GenerativeModel, Part

In [ ]:
pd.set_option("display.max_rows", None)
os.chdir("/repo")

## files

There are three folders of interest:
- texts: where the images and their various transcribtions are stored
- keys: authentication tokens for OpenAI, Google, and Anthropic
- analysis: pickled dataframe of all infernced and difference calculations and a visualization of them

In [ ]:
!ls ./data/

In [ ]:
!ls ./data/keys/

In [ ]:
!ls ./data/texts/

In [ ]:
!ls ./data/texts/images

## global vars

Here all global variables, such as paths or various models, api connections are defined

In [ ]:
# file paths
texts_folder = Path("./data/texts/")
images_folder = Path("./data/texts/images/")
scores_folder = Path("./data/scores/")
analysis_plot_metrics_per_workflow_path = Path("./data/analysis/plot_metrics_per_workflow.png")
analysis_plot_metrics_per_image_path = Path("./data/analysis/plot_metrics_per_image.png")
analysis_plot_metrics_variances_path = Path("./data/analysis/plot_metrics_variances.png")
readme_path = Path("./README.md")

In [ ]:
# keys and setup for AI APIs

# google_token_path = Path("./data/keys/google_key.json")
# other_token_path = Path("./data/keys/tokens.json")

# # tokens
# with open(other_token_path, "r") as f:
#     tokens_dict = json.load(f)

# # openai
# openai_client = openai.OpenAI(api_key=tokens_dict["openai_token"])

# # google
# google_credentials = service_account.Credentials.from_service_account_file(google_token_path)
# google_vision_client = vision.ImageAnnotatorClient(credentials=google_credentials)
# vertexai.init(project="project-pressmint-ocr", location="us-central1", credentials=google_credentials)
# google_model = GenerativeModel("gemini-2.5-flash-lite")

# # anthropic
# anthropic_client = anthropic.Anthropic(api_key=tokens_dict["anthropic_token"])

## ocr_workflows

In [ ]:
ocr_workflow_list = [
    "anno",
    # "anno_openai_one_shot",
    # "anno_openai_three_shot",
    # "anno_openai_two_shot",
    # "anno_openai_zero_shot",
    "anthropic_1_simple",
    # "anthropic_2_extensive", # didn't work; response timed out
    "churro_1_simple",
    "churro_2_german_extensive_2",
    "churro_3_english_extensive_3",
    "churro_4_one_shot",
    "churro_5_one_shot_s_replaced",
    "churro_6_two_shot",
    "churro_7_two_shot_prompts_adapted",
    "churro_8_two_shot_zero_temperature",
    "churro_9_two_shot_zero_temperature",
    "deepseek_ocr_1_default",
    "deepseek_ocr_2_german_extensive_2",
    "deepseek_ocr_3_english_extensive_2",
    "dots_ocr_1_default",
    "dots_ocr_3_default",
    "dots_ocr_4_default",
    # "dots_ocr_5_german_extensive", # empty output
    "dots_ocr_6_english_extensive",
    "dots_ocr_7_german_extensive_2",
    "dots_ocr_8_one_shot",
    "dots_ocr_9_three_shot",
    "dots_ocr_10_one_shot_english_extensive",
    "dots_ocr_11_three_shot_english_extensive",
    "google_gemini_1_simple",
    "google_gemini_2_extensive",
    # "google_gemini_3_one_shot_simple", # didn't work; response timed out
    "google_vision",
    "openai_1_simple",
    "openai_2_extensive",
    "openai_3_one_shot_simple",
    "pero_ocr",
    "pero_scribblesense",
    "transkribus",
    "transkribus_corrected",
    # "tsk_openai_one_shot",
    # "tsk_openai_three_shot",
    # "tsk_openai_two_shot",
    # "tsk_openai_zero_shot",
]

# text comparison metrics

The different metrics that are used:

- WER (lower = better)
- CER (lower = better)
- python's native difflib (higher = better)

## diff_wer

In [ ]:
def diff_wer(text_a: str, text_b: str) -> float:
    return wer(text_a, text_b)

## diff_cer

In [ ]:
def diff_cer(text_a: str, text_b: str) -> float:
    return cer(text_a, text_b)

## diff_difflib

In [ ]:
def diff_difflib(text_a: str, text_b: str) -> float:
    seq = difflib.SequenceMatcher(None, text_a, text_b, autojunk=False)
    return seq.ratio()

## diff_all

In [ ]:
def diff_all(text_a: str, text_b: str) -> float:
    return {
        "WER": diff_wer(text_a, text_b),
        "CER": diff_cer(text_a, text_b),
        "difflib": diff_difflib(text_a, text_b),
    }

## tests

Here, the difference metrics are tested on small examples to showcase their different approaches.

From this small experiment, it is clear that Levenshtein, WER, CER behave similarily (as they concern characters respective their positioning),
while difflib behaves differently as it finds longest matching substrings recursively.

The former three thus punish misaligned but correct subtexts more than difflib.

Note that Levenshtein, WER, CER indicate worse similarity the higher their values are, while difflib returns 1 if texts are identical and 0 if not at all.

In [ ]:
text_a = """
Lorem ipsum dolor sit amet, consectetur adipiscing elit.
Sed do eiusmod tempor incididunt ut labore et dolore.
"""

In [ ]:
# Letzte Wörter:
# elat statt elit
# dalare statt dolore
text_b = """
Lorem ipsum dolor sit amet, consectetur adipiscing elat.
Sed do eiusmod tempor incididunt ut labore et dalare.
"""

In [ ]:
# o mit a
# i mit e
# d mit y
text_c = """
Larem epsum yalar set amet, cansectetur ayepesceng elet.
Sey ya eeusmay tempar enceyeyunt ut labare et yalare.
"""

In [ ]:
# Erster und zweiter Satz vertauscht
text_d = """
Sed do eiusmod tempor incididunt at labore dolore.
Lorem ipsum dolor sit amet consectetur adipiscing elit.
"""

In [ ]:
# Gänzlich neue Sätze
text_e = """
Commodo nulla facilisi nullam vehicula ipsum a arcu.
Pulvinar proin gravida hendrerit lectus.
"""

In [ ]:
diff_all(text_a, text_a)

In [ ]:
diff_all(text_a, text_b)

In [ ]:
diff_all(text_a, text_c)

In [ ]:
diff_all(text_a, text_d)

In [ ]:
diff_all(text_a, text_e)

# reader functions

Various helper functions to read data. Note that in the context of this notebook analysis, it is assumed that images have unique ids
(e.g. `1917-12-28_1`) which are consistently used across all inferences. This means, that for the image `1917-12-28_1.jpg` there will
be corresponding `1917-12-28_1.txt` in each inference subfolder (e.g. `google_vision`).

Also, different OCR tools / LLMs require their images to be submitted in different formats. Such functions are defined respectively.

## get_image_path

In [ ]:
def get_image_path(image_id: str) -> Path:
    return images_folder / Path(image_id + ".jpg")

## read_ocr_text

In [ ]:
def read_ocr_text(ocr_workflow: str, image_id: str) -> str:
    with open(texts_folder / Path(ocr_workflow) / Path(image_id + ".txt"), "r") as f:
        return f.read()

## read_image_as_string

In [ ]:
def read_image_as_string(image_id: str) -> str:
    with open(get_image_path(image_id), "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

## read_image_as_binary

In [ ]:
def read_image_as_binary(image_id: str) -> bytes:
    with open(get_image_path(image_id), "rb") as f:
        return f.read()

## read_image_as_gemini_part

In [ ]:
def read_image_as_gemini_part(image_id: str) -> bytes:
    img = Image.open(get_image_path(image_id))
    img_byte_arr = io.BytesIO()
    img.save(img_byte_arr, format="JPEG")
    img_byte_arr = img_byte_arr.getvalue()
    image_part = Part.from_data(data=img_byte_arr, mime_type="image/jpeg")
    return image_part

## get_image_ids

In [ ]:
def get_image_ids() -> list[str]:
    image_id_list = []
    for image_path in sorted(images_folder.iterdir()):
        # image 1914-06-29_6 was added later, but since there is no inference for it 
        # and doing it again is not worth it, it is skipped
        if not str(image_path).endswith("1914-06-29_6.jpg"):
            image_id_list.append(image_path.stem)
    return image_id_list

In [ ]:
for image_id in get_image_ids():
    print(image_id)
    print(len(read_image_as_string(image_id)))

# OCR inferences

Here, the major OCR inference functions are defined to be reused in aggregate logic or individual experiments.

A few rely on a `_base` function which encompasses their core logic, and then have corresponding wrapper functions for
ease of calling in the aggregate analysis.

In [ ]:
ocr_inference_functions_dict = {}

## prompts

In [ ]:
prompt_simple = "Extrahiere den gesamten Text aus diesem Bild."
prompt_extensive = (
    "Das ist ein Scan einer deutschen historischen Zeitung aus dem frühen 20. Jahrhundert."
    "Bitte führe OCR darauf aus. "
    "Beachte dabei, dass die Schrift in Fraktur gehalten ist. "
    "Versuche keine Interpretationen zu machen bezüglich der Wörter, sondern transkripiere jeden Buchstaben wie du ihn siehst. "
    "Ohne irgendwelche Metabeschreibungen, nur den Text alleine."
)
prompt_one_shot_ground_truth = "Hier ist ein Beispielbild einer historischen Frakturschrift-Zeitung mit seiner korrekten Transkription:"
prompt_one_shot_explanation = "Bitte beschreibe das folgende ähnliche Bild, so gut wie möglich."
prompt_one_shot_ocr_inference = "Anhand von dem vorher gezeigten Beispielbild, extrahiere den Text aus dem folgenden Bild."

## openai_base

Base function for OpenAI OCR inference. Hard-coded to `GPT-4o`

In [ ]:
def openai_base(prompt: str, image_id: str) -> str:
    image = read_image_as_string(image_id)
    response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": prompt,
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{image}",
                            "detail": "high",
                        },
                    },
                ],
            }
        ],
        max_tokens=2048,
    )
    return response.choices[0].message.content

## openai_1_simple

Reuses the logic of `openai_base` but with a simple prompt.

In [ ]:
def openai_1_simple(image_id: str) -> str:
    return openai_base(prompt_simple, image_id)


ocr_inference_functions_dict["openai_1_simple"] = openai_1_simple

## openai_2_extensive

Reuses the logic of `openai_base` but with a more extensive prompt.

In [ ]:
def openai_2_extensive(image_id: str) -> str:
    return openai_base(prompt_extensive, image_id)


ocr_inference_functions_dict["openai_2_extensive"] = openai_2_extensive

## openai_3_one_shot_simple

Core function for One Shot inference with OpenAI. This function needs two prompts and two image ids. One prompt and image id for ground truth / gold data to prime the model, and the other for inference.

Note, that currently this still fails, with unknown reasons as chatgpt just returns texts like "I'm sorry, I can not assist in this", without any elaboration. Various different aprpoaches in prompting failed too.

In [ ]:
def openai_3_one_shot_simple(prompt_ground_truth: str, prompt_inference: str, image_id_ground_truth: str, image_id_inference: str) -> str:
    image_ground_truth = read_image_as_string(image_id_ground_truth)
    image_inference = read_image_as_string(image_id_inference)
    text_ground_truth = read_ocr_text(transkribus_corrected_folder, image_id)
    response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": prompt_ground_truth,
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{image_ground_truth}",
                            "detail": "high",
                        },
                    },
                    {"type": "text", "text": text_ground_truth},
                    {"type": "text", "text": prompt_inference},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{image_inference}",
                            "detail": "high",
                        },
                    },
                ],
            }
        ],
        max_tokens=2048,
    )
    return response.choices[0].message.content


ocr_inference_functions_dict["openai_3_one_shot_simple"] = openai_3_one_shot_simple

## openai_one_shot_simple

In [ ]:
def openai_one_shot_simple(image_id_ground_truth: str, image_id_inference: str):
    return openai_3_one_shot_simple(prompt_one_shot_ground_truth, prompt_one_shot_explanation, image_id_ground_truth, image_id_inference)


ocr_inference_functions_dict["openai_one_shot_simple"] = openai_one_shot_simple

## google_vision

Google Cloud Vision API is not a LLM or Gemini, but an image analyser, which can be used for OCR too.

In [ ]:
def google_vision(image_id: str) -> str:
    image = vision.Image(content=read_image_as_binary(image_id))
    response = google_vision_client.document_text_detection(image=image)
    document = response.full_text_annotation
    return document.text


ocr_inference_functions_dict["google_vision"] = google_vision

## google_gemini_base

In [ ]:
def google_gemini_base(prompt: str, image_id: str) -> str:
    image_part = read_image_as_gemini_part(image_id)
    response = google_model.generate_content([prompt, image_part])
    return response.text

## google_gemini_1_simple

In [ ]:
def google_gemini_1_simple(image_id: str) -> str:
    return google_gemini_base(prompt_simple, image_id)


ocr_inference_functions_dict["google_gemini_1_simple"] = google_gemini_1_simple

## google_gemini_2_extensive

In [ ]:
def google_gemini_2_extensive(image_id: str) -> str:
    return google_gemini_base(prompt_extensive, image_id)


ocr_inference_functions_dict["google_gemini_2_extensive"] = google_gemini_2_extensive

## google_gemini_one_shot_base

In [ ]:
def google_gemini_one_shot_base(
    prompt_ground_truth: str, prompt_inference: str, image_id_ground_truth: str, image_id_inference: str
) -> str:
    image_ground_truth = read_image_as_gemini_part(image_id_ground_truth)
    image_inference = read_image_as_gemini_part(image_id_inference)
    contents = [prompt_ground_truth, image_ground_truth, prompt_inference, image_inference]
    response = google_model.generate_content(contents)
    return response.text

## google_gemini_3_one_shot_simple

In [ ]:
def google_gemini_3_one_shot_simple(image_id_ground_truth: str, image_id_inference: str):
    return google_gemini_one_shot_base(
        prompt_one_shot_ground_truth, prompt_one_shot_ocr_inference, image_id_ground_truth, image_id_inference
    )


ocr_inference_functions_dict["google_gemini_3_one_shot_simple"] = google_gemini_3_one_shot_simple

## anthropic_base

In [ ]:
def anthropic_base(prompt: str, image_id: str) -> str:
    image = read_image_as_string(image_id)
    message = anthropic_client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=4000,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": "image/jpeg",
                            "data": image,
                        },
                    },
                    {"type": "text", "text": prompt},
                ],
            }
        ],
    )
    return message.content[0].text

## anthropic_1_simple

In [ ]:
def anthropic_1_simple(image_id: str) -> str:
    return anthropic_base(prompt_simple, image_id)


ocr_inference_functions_dict["anthropic_1_simple"] = anthropic_1_simple

## anthropic_2_extensive

In [ ]:
def anthropic_2_extensive(image_id: str) -> str:
    return anthropic_base(prompt_extensive, image_id)


ocr_inference_functions_dict["anthropic_2_extensive"] = anthropic_2_extensive

## get_all_ocr_func

In order to bulk process, the OCR functions above are bundled into a list which will be iterated over and each function will be called.

In [ ]:
def get_all_ocr_func() -> list[Callable]:
    return list(ocr_inference_functions_dict.values())


for f in get_all_ocr_func():
    print(f.__name__, ":", f)

## tests

Tests all functions defined in the list above. Is disabled by default to avoid unncessary API costs.

In [ ]:
# change this boolean to run the tests.
enable_test_all = False
if enable_test_all:
    for ocr_func in get_all_ocr_func():
        print("\n-----------------------------------------------------------------------")
        print(ocr_func.__name__, "\n")
        if ocr_func in [openai_one_shot_simple, google_gemini_3_one_shot_simple]:
            text = ocr_func("1915-12-28_1", "1915-12-28_2")
        else:
            text = ocr_func("1915-12-28_1")
        print(text[:500])

# transform dots.ocr and deepseek-ocr

## transform_dots_ocr_1-4_default

In [ ]:
enable_transformation = False
if enable_transformation:
    pkl_folder = texts_folder / Path("dots_ocr_1-4_default_pkl")
    for pkl_file_name in os.listdir(pkl_folder):
        with open(pkl_folder / Path(pkl_file_name), "rb") as pkl_file:
            print(f"{pkl_file_name=}")
            dots_ocr_output_pkl = pickle.load(pkl_file)
            if pkl_file_name.endswith("1.pkl"):
                continue
            elif pkl_file_name.endswith("0.pkl") or pkl_file_name.endswith("3.pkl"):
                try:
                    dots_ocr_output = eval(dots_ocr_output_pkl[0])
                except Exception as ex:
                    print(f"WARNING: could not transform: {pkl_file_name} due to Exception: {ex}")
                    continue
                text = ""
                for dots_ocr_output_part in dots_ocr_output:
                    text_part = dots_ocr_output_part.get("text")
                    if text_part is not None:
                        text += text_part
            elif pkl_file_name.endswith("2.pkl"):
                text = dots_ocr_output_pkl[0]
            dots_ocr_output_folder = None
            if pkl_file_name.endswith("0.pkl"):
                dots_ocr_output_folder = Path("dots_ocr_1_default")
            # elif pkl_file_name.endswith("1.pkl"):
            #     dots_ocr_output_folder = Path("dots_ocr_2_default")
            elif pkl_file_name.endswith("2.pkl"):
                dots_ocr_output_folder = Path("dots_ocr_3_default")
            elif pkl_file_name.endswith("3.pkl"):
                dots_ocr_output_folder = Path("dots_ocr_4_default")
            dots_ocr_out_file_name = pkl_file_name[:-6] + ".txt"
            print(f"{dots_ocr_out_file_name=}")
            with open(texts_folder / dots_ocr_output_folder / Path(dots_ocr_out_file_name), "w") as dots_ocr_out_file:
                dots_ocr_out_file.write(text)

## transform dots_ocr adapted

In [ ]:
enable_transformation = True
if enable_transformation:
    for all_output_folder, target_folder in [
        ("dots_ocr_6_english_extensive_pkl", "dots_ocr_6_english_extensive"),
        ("dots_ocr_7_german_extensive_2_pkl", "dots_ocr_7_german_extensive_2"),
        ("dots_ocr_8_one_shot_pkl", "dots_ocr_8_one_shot"),
        ("dots_ocr_9_three_shot_pkl", "dots_ocr_9_three_shot"),
        ("dots_ocr_10_one_shot_english_extensive_pkl", "dots_ocr_10_one_shot_english_extensive"),
        ("dots_ocr_11_three_shot_english_extensive_pkl", "dots_ocr_11_three_shot_english_extensive"),
    ]:
        pkl_folder = texts_folder / Path(all_output_folder)
        for pkl_file_name in os.listdir(pkl_folder):
            with open(pkl_folder / Path(pkl_file_name), "rb") as pkl_file:
                print(f"{pkl_file_name=}")
                dots_ocr_output_pkl = pickle.load(pkl_file)
                text = dots_ocr_output_pkl[0]
                dots_ocr_out_file_name = pkl_file_name.replace(".pkl", ".txt")
                print(f"{dots_ocr_out_file_name=}")
                with open(texts_folder / Path(target_folder) / Path(dots_ocr_out_file_name), "w") as dots_ocr_out_file:
                    dots_ocr_out_file.write(text)

## transform_deepseek_ocr

In [ ]:
enable_deepseepk_ocr_transformation = False
if enable_deepseepk_ocr_transformation:
    for all_output_folder, target_folder in [
        ("deepseek_ocr_1_default_all_output", "deepseek_ocr_1_default"),
        ("deepseek_ocr_2_german_extensive_2_all_output", "deepseek_ocr_2_german_extensive_2"),
        ("deepseek_ocr_3_english_extensive_2_all_output", "deepseek_ocr_3_english_extensive_2"),
    ]:
        all_output_folder = Path(texts_folder) / Path(all_output_folder)
        target_folder = Path(texts_folder) / Path(target_folder)
        for image_id in os.listdir(all_output_folder):
            with open(all_output_folder / Path(image_id) / Path("result.mmd"), "r") as f:
                print(f"{image_id=}")
                text = f.read()
                with open(target_folder / Path(image_id + ".txt"), "w") as deepseek_ocr_out_file:
                    deepseek_ocr_out_file.write(text)

# aggregated analysis

Here, all previously modular functions come together.

All OCR functions are called on all images and their respective outputs are persisted as well as compared against the provided ground truth to evaluate their quality.

Note a few OCR inferences are done outside of this notebook, such as with dots.ocr or deepseek-ocr, as they require CLIP GPUs. Their respective functions just load the pre-computed inference data.

## inference all

In [ ]:
# change this boolean to run the entire chain. Is disabled by default to avoid unncessary API costs.
enable_inference_all = False
if enable_inference_all:

    # iterate over functions
    for ocr_func in get_all_ocr_func():
        print(f"ocr_func: {ocr_func.__name__}")

        # iterate over images
        image_id_list = get_image_ids()
        for image_id in image_id_list:
            print(f"image_id: {image_id}")

            # do inference
            if ocr_func in [openai_one_shot_simple, google_gemini_3_one_shot_simple]:
                image_id_ground_truth = None
                if image_id.endswith("1"):
                    image_id_ground_truth = image_id[:-1] + "2"
                elif image_id.endswith("2"):
                    image_id_ground_truth = image_id[:-1] + "1"
                elif image_id.endswith("3"):
                    image_id_ground_truth = image_id[:-1] + "4"
                elif image_id.endswith("4"):
                    image_id_ground_truth = image_id[:-1] + "3"
                elif image_id.endswith("5"):
                    image_id_ground_truth = image_id[:-1] + "2"
                elif image_id.endswith("6"):
                    image_id_ground_truth = image_id[:-1] + "1"
                elif image_id.endswith("7"):
                    image_id_ground_truth = image_id[:-1] + "3"
                text_inferred = ocr_func(image_id_ground_truth, image_id)
            else:
                text_inferred = ocr_func(image_id)

            # write inferenced text
            ocr_output_folder = get_ocr_output_folder(ocr_func)
            with open(ocr_output_folder / Path(image_id + ".txt"), "w") as f:
                f.write(text_inferred)

## compare all

In [ ]:
enable_compare_all = True
if enable_compare_all:
    diff_name_list = ["WER", "CER", "difflib"]
    diff_data_columns = ["workflow", "image_id"] + diff_name_list
    diff_data_all = []

    # iterate over functions
    for ocr_workflow in ocr_workflow_list:
        if ocr_workflow == "transkribus_corrected":
            continue
        print(f"{ocr_workflow=}")

        ocr_workflow_scores_pkl = scores_folder / Path(ocr_workflow + ".pkl")
        if not ocr_workflow_scores_pkl.exists():
            print("calculating")

            # prepare average data dict
            diff_sum_dict = {diff_name: 0 for diff_name in diff_name_list}

            # iterate over images
            ocr_workflow_scores = []
            compared_count = 0
            for image_id in get_image_ids():
                print(f"{image_id=}")

                # load ground truth and inferenced texts
                text_ground_truth = read_ocr_text("transkribus_corrected", image_id)
                try:
                    text_inferred = read_ocr_text(ocr_workflow, image_id)
                except Exception as ex:
                    print(f"Exception when reading inferred text: {ex}")
                    continue
                diff_dict = diff_all(text_ground_truth, text_inferred)

                # append results
                row = [ocr_workflow, image_id]
                for diff_name in diff_name_list:
                    diff_sum_dict[diff_name] = diff_sum_dict[diff_name] + diff_dict[diff_name]
                    row.append(diff_dict[diff_name])
                compared_count += 1
                ocr_workflow_scores.append(row)

            # append average
            row = [ocr_workflow, "average"]
            for diff_name in diff_name_list:
                row.append(diff_sum_dict[diff_name] / compared_count)
            ocr_workflow_scores.append(row)

            # persist scores
            with open(ocr_workflow_scores_pkl, "wb") as f:
                pickle.dump(ocr_workflow_scores, f)

        else:
            # load pre-calculated scores
            print("loading pre-calculated scores")
            with open(ocr_workflow_scores_pkl, "rb") as f:
                ocr_workflow_scores = pickle.load(f)

        diff_data_all.extend(ocr_workflow_scores)

    df = pd.DataFrame(diff_data_all, columns=diff_data_columns)

## results full

In [ ]:
df

## results per image

In [ ]:
scores_per_image = []
for image_id in get_image_ids():
    df_per_image_current = df[df["image_id"] == image_id]
    scores_per_image_current = [image_id]
    for metric in ["WER", "CER", "difflib"]:
        avg = sum(df_per_image_current[metric]) / len(df_per_image_current)
        scores_per_image_current.append(avg)
    scores_per_image.append(scores_per_image_current)

df_image = pd.DataFrame(scores_per_image, columns=["image_id", "WER", "CER", "difflib"])
df_image

## calculate variance

In [ ]:
df_var = []
for ocr_workflow in ocr_workflow_list:
    if ocr_workflow == "transkribus_corrected":
        continue
    df_w = df[df["workflow"] == ocr_workflow]
    df_var.append(
        (
            ocr_workflow,
            df_w["WER"].var(),
            df_w["CER"].var(),
            df_w["difflib"].var(),
        )
    )
df_var = pd.DataFrame(df_var, columns=["workflow", "WER", "CER", "difflib"])
df_var

## rank results

In [ ]:
def rank(df, invert_difflib, columns):
    first = True
    for r in df.iterrows():
        r = r[1]
        if first:
            wer_min = r["WER"]
            wer_max = r["WER"]
            cer_min = r["CER"]
            cer_max = r["CER"]
            difflib_min = r["difflib"]
            difflib_max = r["difflib"]
            first = False
        else:
            if r["WER"] < wer_min:
                wer_min = r["WER"]
            if r["WER"] > wer_max:
                wer_max = r["WER"]
            if r["CER"] < cer_min:
                cer_min = r["CER"]
            if r["CER"] > cer_max:
                cer_max = r["CER"]
            if r["difflib"] < difflib_min:
                difflib_min = r["difflib"]
            if r["difflib"] > difflib_max:
                difflib_max = r["difflib"]

    df_data_in_processing = []
    for r in df.iterrows():
        r = r[1]
        score = 0
        wer_norm = 1 - ((r["WER"] - wer_min) / (wer_max - wer_min))
        cer_norm = 1 - ((r["CER"] - cer_min) / (cer_max - cer_min))
        if invert_difflib:
            difflib_norm = 1 - ((r["difflib"] - difflib_min) / (difflib_max - difflib_min))
        else:
            difflib_norm = (r["difflib"] - difflib_min) / (difflib_max - difflib_min)
        score = (wer_norm + cer_norm + difflib_norm) / 3
        df_data_in_processing.append([score] + [r[c] for c in columns])
    df_data_in_processing = sorted(df_data_in_processing, key=lambda x: -x[0])
    df_data_in_processing = [t[1:] for t in df_data_in_processing]

    df_ranked = pd.DataFrame(df_data_in_processing, columns=columns)
    return df_ranked


default_columns = ["workflow", "image_id", "WER", "CER", "difflib"]
df_all_ranked = rank(df, invert_difflib=False, columns=default_columns)
df_avg_ranked = rank(df[df["image_id"] == "average"], invert_difflib=False, columns=default_columns)
df_var_ranked = rank(df_var, invert_difflib=True, columns=["workflow", "WER", "CER", "difflib"])
df_img_ranked = rank(df_image, invert_difflib=False, columns=["image_id", "WER", "CER", "difflib"])

## df_all_ranked

In [ ]:
df_all_ranked

## plot

The average rows of each inference are aggreagated and plotted here into a parallel coordinates chart, ideal for multi-dimensional analysis.

In [ ]:
def plot_parallel_coord(df, title, plot_path, flip_wer_cer, flip_difflib, first_column, include_rows=None, exclude_rows=None):

    if include_rows is not None or exclude_rows is not None:
        plot_rows = []
        for row in df.iterrows():
            if include_rows is not None and row[1][first_column] in include_rows:
                plot_rows.append(row[1])
            if exclude_rows is not None and row[1][first_column] not in exclude_rows:
                plot_rows.append(row[1])
        df = pd.DataFrame(plot_rows)

    tick_num_list = list(range(len(df[first_column])))
    dim_list = [
        {
            "label": first_column,
            "values": tick_num_list,
            "tickvals": tick_num_list,
            "ticktext": list(df[first_column]),
            "range": [max(tick_num_list), min(tick_num_list)],
        }
    ]
    for col in ["WER", "CER", "difflib"]:
        dim_dict = {"label": col, "values": list(df[col])}
        if (flip_wer_cer and col in ["WER", "CER"]) or (flip_difflib and col == "difflib"):
            dim_dict["range"] = [max(df[col]), min(df[col])]

        dim_list.append(dim_dict)

    fig = go.Figure(
        data=go.Parcoords(
            line={
                "color": list(df.index)[::-1],
                "colorscale": "Rainbow",
            },
            dimensions=dim_list,
        )
    )
    fig.update_layout(
        height=1000,
        font=dict(size=14),
        margin=dict(l=200),
        title=title,
    )
    fig.write_image(plot_path, width=800, height=1000, scale=2)
    fig.show()


plot_parallel_coord(
    df_avg_ranked,
    title="scores per workflow",
    plot_path=analysis_plot_metrics_per_workflow_path,
    flip_wer_cer=True,
    flip_difflib=False,
    first_column="workflow",
)
plot_parallel_coord(
    df_var_ranked,
    title="variance per metric",
    plot_path=analysis_plot_metrics_variances_path,
    flip_wer_cer=True,
    flip_difflib=True,
    first_column="workflow",
)
plot_parallel_coord(
    df_img_ranked,
    title="scores per image",
    plot_path=analysis_plot_metrics_per_image_path,
    flip_wer_cer=True,
    flip_difflib=False,
    first_column="image_id",
    # exclude_rows=["churro_1_simple", "deepseek_ocr_1_default", "dots_ocr_3_default", "dots_ocr_6_english_extensive", "deepseek_ocr_2_german_extensive_2"],
)

## write results to readme

In [ ]:
readme_content = ""
with open(readme_path, "r") as f:
    for line in f:
        readme_content += line
        if line == "## comparison results details\n":
            readme_content += "\n"
            break


readme_content += df.to_markdown()
readme_content += "\n\n## variance details\n\n"
readme_content += df_var_ranked.to_markdown()
readme_content += "\n\n## results per image\n\n"
readme_content += df_img_ranked.to_markdown()
readme_content += "\n\n## all image results ranked\n\n"
readme_content += df_all_ranked.to_markdown()
with open(readme_path, "w") as f:
    f.write(readme_content)